In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer, from_unixtime
from delta.tables import DeltaTable

################################################
############--- ordered_products ---############
################################################

def ordered_products (df):
    ordered_products_schema = ArrayType(
        StructType([
            StructField("curr", StringType()),
            StructField("id", StringType()),
            StructField("name", StringType()),
            StructField("price", StringType()),
            StructField("promotion_info", StringType()),
            StructField("qty", StringType()),
            StructField("unit", StringType())
        ])
    )

    df_ordered_products = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_products_schema)) \
                    .withColumn("ordered_products", explode_outer(col("ordered_products"))) \
                        .select(
                                "customer_id", 
                                "customer_name",
                                "order_number",
                                "ordered_products.id",
                                col("ordered_products.name").alias("product_name"),
                                "ordered_products.price",
                                "ordered_products.curr",
                                # "ordered_products.promotion_info",
                                "ordered_products.qty",
                                "ordered_products.unit",
                                from_unixtime(col("order_datetime")).alias("order_timestamp")
                                )
    return df_ordered_products

##########################################
############--- promotions ---############
##########################################

def promotions (df):
    df_promo_info_schema = ArrayType(
        StructType([
            StructField("promo_disc", StringType()),
            StructField("promo_id", StringType()),
            StructField("promo_item", StringType()),
            StructField("promo_qty", StringType())
        ])
    )

    df_promotions = df.withColumn("promo_info", from_json(col("promo_info"), df_promo_info_schema)) \
                        .withColumn("promo_info", explode_outer("promo_info")) \
                            .filter(col("promo_info").isNotNull()) \
                                .select(
                                        "customer_id",
                                        "order_number",
                                        col("promo_info.promo_id").alias("promo_id"),
                                        col("promo_info.promo_item").alias("promo_product_id"),
                                        col("promo_info.promo_disc").alias("discount"),
                                        col("promo_info.promo_qty").cast("int").alias("promo_quantity"),
                                        from_unixtime(col("order_datetime")).alias("order_timestamp")
                                        ) \
                                .dropDuplicates()
    return df_promotions

##########################################
############--- cliked_items ---############
##########################################

def clicked_items (df):
    clicked_items_schema = ArrayType(
        ArrayType(StringType())
    )
    df_clicked_items = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_items_schema)) \
                    .withColumn("clicked_items", explode_outer(col("clicked_items"))) \
                        .select(
                                "customer_id", 
                                "customer_name",
                                "order_number",
                                col("clicked_items")[0].alias("product_id"),
                                col("clicked_items")[1].cast("double").alias("score"),
                                from_unixtime(col("order_datetime")).alias("order_timestamp"),
                                )
    return df_clicked_items

##########################################
############--- silver_sales_orders ---#############
##########################################

def silver_sales_orders(df):
    df_sales_orders = df.select(
        "customer_id", 
        "customer_name",
        "order_number",
        "number_of_line_items",
        from_unixtime(col("order_datetime")).alias("order_timestamp"),
    )
    return df_sales_orders

##################################
#####--SCD-1 IMPLEMENTATION--#####
##################################

def scd_merge_table(spark, source_table, target_table, business_key):

    # source_df = spark.table(source_table)

    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table", target_table)
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print("First Load: Silver Table Created - ", target_table)

    else:
        print("Incremental Load: Performing SCD type 1 Merge on ", target_table) 
        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = "AND".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )
        ## ouptput of above merge_condition:
        ## e.g. ["target.customer_id = source.customer_id" and "target.product_id = source.product_id" etc.]

        delta_table.alias("target").merge(source_table.alias("source"),merge_condition,)\
                                        .whenMatchedUpdateAll()\
                                        .whenNotMatchedInsertAll()\
                                        .execute()
        print("Incremental Load: SCD type 1 Merge Completed on ", target_table)

##########################################
############--- MAIN CODE ---#############
##########################################


bronze_table = "ecommerce_analytics.bronze.sales_orders"
print(f"Reading bronze layer table: sales_orders")
df = spark.read.table(bronze_table)

# Transformation 
print(f"Data transformation started on sales_orders...")
ordered_products_df = ordered_products(df)
promotions_df = promotions(df)
clicked_items_df = clicked_items(df)
sales_orders_df = silver_sales_orders(df)

# Write Silver Tables (Performing SCD 1)
scd_merge_table(spark, ordered_products_df, "ecommerce_analytics.silver.ordered_products", ["order_number","id","price"])
scd_merge_table(spark, promotions_df, "ecommerce_analytics.silver.promotions", ["order_number","promo_id","promo_quantity","promo_product_id","order_timestamp"])
scd_merge_table(spark, clicked_items_df, "ecommerce_analytics.silver.clicked_items", ["order_number","product_id"])
scd_merge_table(spark, sales_orders_df, "ecommerce_analytics.silver.silver_sales_orders", ["order_number","number_of_line_items"])

print(f"Finished: Write completed for all silver layer tables")

'''
print(f"Processing data - Transforming ordered_products column of sales_orders")
df_ordered_products = ordered_products(df)
print(f"Writing silver layer table: order_products")
df_ordered_products.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.order_products")

print(f"Processing data - Transforming promo_info column of sales_orders")
df_promotions = promotions(df)
print(f"Writing silver layer table: promotions")
df_promotions.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.promotions")

print(f"Processing data - Transforming clicked_items column of sales_orders")
df_clicked_items = clicked_items(df)
print(f"Writing silver layer table: clicked_items")
df_clicked_items.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.clicked_items")

print(f"Processing data - Selecting & Transforming remaining column of sales_orders")
df_sales_orders = silver_sales_orders(df)
print(f"Writing silver layer table: silver_sales_orders")
df_sales_orders.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.silver_sales_orders")

print(f"Finished: Write completed for all silver layer tables")
'''

# Process

##### Steps: 
1. read the column
2. define schema in StructType and StructField 
3. parse the schema using from_json function 
4. explode the parsed cloumn
5. combine the code
6. write it into silver schema

In [0]:
'''
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer

df = spark.read.table("ecommerce_analytics.bronze.sales_orders")

########################################
############---promotion---############
########################################

def promotion (df):
    df_promo_info_schema = ArrayType(
        StructType([
            StructField("promo_disc", StringType()),
            StructField("promo_id", StringType()),
            StructField("promo_item", StringType()),
            StructField("promo_qty", StringType())
        ])
    )

    df_promo = df.withColumn("promo_info", from_json(col("promo_info"), df_promo_info_schema)) \
                        .withColumn("promo_info", explode_outer("promo_info")) \
                            .filter(col("promo_info").isNotNull()) \
                                .select(
                                        "customer_id",
                                        "order_number",
                                        col("promo_info.promo_id").alias("promo_id"),
                                        col("promo_info.promo_item").alias("promo_product_id"),
                                        col("promo_info.promo_disc").alias("discount"),
                                        col("promo_info.promo_qty").cast("int").alias("promo_quantity")
                                        )
    return df_promo

# df_promo.display()

df_promo.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.promotions")
''''''


In [0]:
'''# reading sales_orders table (bronze layer) for performing transformation on required columns and writing to silver layer as a table
 
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()
'''

In [0]:
df.printSchema()

### Nested data columns

**-> Array of Array (clicked_items columns) e.g.:** 

```
  [
    ["AVpgIu4Q1cnluZ0-xBK-","13"],
    ["AVpfeG5oilAPnD_xcTsG","27"],
    ["AVqVGaEBv8e3D1O-ldFu","64"],
    ["AVpg-Wj61cnluZ0-8sZe","87"],
    ["AVphTO5W1cnluZ0-Aygg","52"],
    ["AVpfMVD-ilAPnD_xW6bu","49"]
  ]
```

**-> Array of Struct (ordered_products columns) e.g.:** 

```
  [
    {"curr":"USD","id":"AVphTO5W1cnluZ0-Aygg","name":"Adventura SH 140 II Shoulder Bag (Black)","price":"27","promotion_info":null,"qty":"1","unit":"pcs"},
    {"curr":"USD","id":"AVpfMVD-ilAPnD_xW6bu","name":"Rony - BC-TRX Battery Charger - Black","price":"31","promotion_info":
      {"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"2"},
    "qty":"2","unit":"pcs"}
  ]
```

**-> Array of Struct (promo_info columns) e.g.:**

```
  [
    {"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"2"}
  ]
```

> Three most important functions :
- from_json - (converts string to structured columns)
- explode - (create new rows but cannot handles NULL)
- explode_outer - (create new row and can handles NULL)

> ArrayType -> StructType And StructField 

In [0]:
'''from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

ordered_products_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ])
)

df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_products_schema))
'''

In [0]:
df_parsed.printSchema()

In [0]:
'''from pyspark.sql.functions import explode_outer

df_exploded_op = df_parsed.withColumn("ordered_products", explode_outer(col("ordered_products")))

df_exploded_op.display()'''

In [0]:
'''
df_ordered_products = df_exploded_op.select(
    "customer_id", 
    "customer_name",
    "order_number",
    "ordered_products.id",
    col("ordered_products.name").alias("product_name"),
    "ordered_products.price",
    "ordered_products.curr",
    # "ordered_products.promotion_info",
    "ordered_products.qty",
    "ordered_products.unit",
)

df_ordered_products.display()
'''

# Process

##### Steps: 
1. read the column
2. define schema in StructType and StructField 
3. parse the schema using from_json function 
4. explode the parsed cloumn
5. combine the code
6. write it into silver schema

In [0]:
'''# Step 1 - reading nested column

df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
display(df.select("clicked_items"))'''

In [0]:
# df.printSchema()

In [0]:
'''# Step 2 - Define Schema

from pyspark.sql.types import ArrayType, StringType

clicked_items_schema = ArrayType(
    ArrayType(StringType())
)'''

In [0]:
'''# Step 3 - Parse schema, i.e. Read nested column with schema

from pyspark.sql.functions import from_json, col

df_parsed = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_items_schema))

# df_parsed.printSchema()
df_parsed.display()
'''

In [0]:
'''# Step 4 - Explode the nested parsed column

from pyspark.sql.functions import explode_outer, col

df_exploded = df_parsed.withColumn("clicked_items", explode_outer(col("clicked_items")))

df_exploded.display()
'''

In [0]:
'''
index [0] -> product_id
index [1] -> score
#############################
# Step 5 - Wrte transformed data to Silver Table

from pyspark.sql.functions import col

clicked_items = df_exploded.select(
    "customer_id", 
    "customer_name",
    "order_number",
    col("clicked_items")[0].alias("product_id"),
    col("clicked_items")[1].cast("double").alias("score")
)
# clicked_items.display()

clicked_items.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.clicked_items")   
'''

In [0]:
'''df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()'''

In [0]:
# Create a final table consiting of remaining columns except metadata columns

'''from pyspark.sql.functions import from_unixtime

def silver_sales_orders(df):
    silver_sales_orders = df.select(
        "customer_id", 
        "customer_name",
        "order_number",
        "number_of_line_items",
        from_unixtime(col("order_datetime")).alias("order_timestamp"),
    )
    return silver_sales_orders

silver_sales_orders.write.mode("overwrite").saveAsTable("ecommerce_analytics.silver.sales_orders")'''